# Day 3 - 대화형 AI, 즉 챗봇!

In [1]:
# 임포트

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# .env 파일에서 환경 변수 로드
# 디버깅용으로 키 접두사 출력

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API 키 있음, 접두사: {openai_api_key[:8]}")
else:
    print("OpenAI API 키 없음")

OpenAI API 키 있음, 접두사: sk-proj-


In [3]:
# 초기화

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [4]:
# 랩 동안 실험적으로 이 전역 변수를 바꿀 예정

system_message = "당신은 도움이 되는 어시스턴트입니다"

## 새 콜백 함수 작성하기

다음 함수를 작성합니다:

`chat(message, history)`

Gradio에 넘겨 줄 **콜백 함수**입니다.

### 이 함수의 역할

사용자 메시지와 이전 대화를 받아서, 응답을 반환합니다.


In [8]:
import gradio as gr
import inspect

print(f"버전: {gr.__version__}")
print(f"설치 경로: {gr.__file__}")

# ChatInterface의 생성자 인자 목록을 직접 확인
signature = inspect.signature(gr.ChatInterface.__init__)
print(f"지원 인자: {list(signature.parameters.keys())}")

버전: 6.9.0
설치 경로: /opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/__init__.py
지원 인자: ['self', 'fn', 'multimodal', 'chatbot', 'textbox', 'additional_inputs', 'additional_inputs_accordion', 'additional_outputs', 'editable', 'examples', 'example_labels', 'example_icons', 'run_examples_on_click', 'cache_examples', 'cache_mode', 'title', 'description', 'flagging_mode', 'flagging_options', 'flagging_dir', 'analytics_enabled', 'autofocus', 'autoscroll', 'submit_btn', 'stop_btn', 'concurrency_limit', 'delete_cache', 'show_progress', 'fill_height', 'fill_width', 'api_name', 'api_description', 'api_visibility', 'save_history', 'validator']


In [9]:
def chat(message, history):
    return "bananas"

In [10]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    return f"당신이 '{message}'라고 했고 이력은 {history}인데, 저는 그래도 bananas라고 말할게요"

In [12]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 조금 나은 채팅 콜백 작성하기

In [13]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [15]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [17]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 계속해 봅시다!

시스템 메시지로 맥락과 예시 답변을 넣기 — 다시 **원샷 프롬핑**입니다.

In [18]:
system_message = "당신은 옷가게의 친절한 점원입니다. 세일 중인 상품을 부드럽게 권유하세요. \
모자는 60% 할인, 그밖 대부분 50% 할인입니다. \
예를 들어 손님이 '모자 사려고요'라고 하면 \
'좋으시겠어요. 모자 종류가 많고 세일 행사 상품도 있어요' 같은 식으로 답하세요. \
뭘 살지 모르겠다면 모자 구매를 권하세요."

In [19]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [20]:
system_message += "\n손님이 신발을 물어보면 오늘 신발은 세일이 아니라고 하고, 모자를 한번 둘러보라고 안내하세요!"

In [21]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [22]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower() or '벨트' in message:
        relevant_system_message += " 이 가게는 벨트를 취급하지 않습니다. 벨트를 요청받으면 세일 중인 다른 상품을 안내하세요."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [23]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">비즈니스 활용</h2>
            <span style="color:#181;">대화형 어시스턴트는 Gen AI의 대표적인 사용 사례이고, 최신 Frontier 모델은 미묘한 대화에 뛰어납니다. Gradio로 UI를 쉽게 만들 수 있고, 프롬프트로 맥락·정보·예시를 주는 방법도 중요한 스킬입니다.
<br/><br/>
여러분 비즈니스에 AI 어시스턴트를 어떻게 적용할지 생각해 보고, 프로토타입을 만들어 보세요. 시스템 프롬프트로 비즈니스 맥락과 LLM 톤을 설정하면 됩니다.</span>
        </td>
    </tr>
</table>